In [ ]:
!pip -q install kaggle
import os
os.makedirs('/root/.kaggle', exist_ok=True)
!cp kaggle.json /root/.kaggle/
!chmod 600 /root/.kaggle/kaggle.json

In [ ]:
import tensorflow as tf
print("GPU Available:", tf.config.list_physical_devices('GPU'))
import os

In [ ]:
dataset_path = r"C:\Users\narra\OneDrive\Desktop\projects\Deep Learning\Pneumonia Detection\chest_xray\chest_xray"

In [ ]:
IMG_SIZE = (224,224)
BATCH_SIZE = 32
SEED = 42

In [ ]:
train_dataset = tf.keras.utils.image_dataset_from_directory(
    os.path.join(dataset_path, "train"),
    validation_split=0.2,
    subset="training",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary"
)

In [ ]:
validation_dataset = tf.keras.utils.image_dataset_from_directory(
    os.path.join(dataset_path, "train"),
    validation_split=0.2,
    subset="validation",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary"
)


In [ ]:
test_dataset = tf.keras.utils.image_dataset_from_directory(
    os.path.join(dataset_path, "test"),
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary",
    shuffle=False
)

In [ ]:
AUTOTUNE=tf.data.AUTOTUNE
normalization=tf.keras.layers.Rescaling(1./255)
data_augmentation=tf.keras.Sequential([
tf.keras.layers.RandomFlip('horizontal'),
tf.keras.layers.RandomRotation(0.1),
tf.keras.layers.RandomZoom(0.1)
])

In [ ]:
train_dataset = train_dataset.map(lambda x,y:(normalization(x),y)).prefetch(AUTOTUNE)
validation_dataset = validation_dataset.map(lambda x,y:(normalization(x),y)).prefetch(AUTOTUNE)
test_dataset = test_dataset.map(lambda x,y:(normalization(x),y)).prefetch(AUTOTUNE)

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D,MaxPooling2D,Flatten,Dense,Dropout,BatchNormalization

In [ ]:
model=Sequential([
data_augmentation,
Conv2D(32,(3,3),padding='same',activation='relu',input_shape=(224,224,3)),
BatchNormalization(),
MaxPooling2D(2,2),
Conv2D(64,(3,3),padding='same',activation='relu'),
BatchNormalization(),
MaxPooling2D(2,2),
Conv2D(128,(3,3),padding='same',activation='relu'),
BatchNormalization(),
MaxPooling2D(2,2),
Flatten(),
Dense(128,activation='relu'),
Dropout(0.5),
Dense(1,activation='sigmoid')
])

In [ ]:
model.compile(optimizer='adam',loss='binary_crossentropy',metrics=['accuracy'])

In [ ]:
model.summary()

In [ ]:
from PIL import Image
import os

dataset_path = r"C:\Users\narra\OneDrive\Desktop\projects\Deep Learning\Pneumonia Detection\chest_xray\chest_xray"

bad_files = []

for root, dirs, files in os.walk(dataset_path):
    for file in files:
        if file.lower().endswith((".jpeg", ".jpg", ".png")):
            path = os.path.join(root, file)
            try:
                img = Image.open(path)
                img.verify()
            except Exception:
                bad_files.append(path)

print("Bad Images:", len(bad_files))

for f in bad_files:
    print(f)

In [ ]:
for file in bad_files:
    os.remove(file)

print("Corrupted images removed.")

In [ ]:
import os

for root, dirs, files in os.walk(dataset_path):
    for file in files:
        path = os.path.join(root, file)

        if os.path.isfile(path):

            if os.path.getsize(path) == 0:
                print("Deleting:", path)
                os.remove(path)

In [ ]:
callbacks=[tf.keras.callbacks.EarlyStopping(monitor='val_loss',patience=5,restore_best_weights=True)]
history=model.fit(train_dataset,validation_data=validation_dataset,epochs=20,callbacks=callbacks)

In [ ]:
import matplotlib.pyplot as plt

plt.plot(history.history['accuracy'],label='Train')
plt.plot(history.history['val_accuracy'],label='Validation')
plt.legend(); plt.show()

plt.plot(history.history['loss'],label='Train')
plt.plot(history.history['val_loss'],label='Validation')
plt.legend(); plt.show()

In [ ]:
from sklearn.metrics import classification_report,confusion_matrix,roc_auc_score

In [ ]:
loss,acc=model.evaluate(test_dataset)
print("Accuracy:",acc)


In [ ]:
y_true=[]
y_prob=[]

In [ ]:
for images,labels in test_dataset:
    pred=model.predict(images,verbose=0)
    y_prob.extend(pred.flatten())
    y_true.extend(labels.numpy().flatten())

In [ ]:
import numpy as np
y_pred=(np.array(y_prob)>0.5).astype(int)
print(classification_report(y_true,y_pred))

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

cm=confusion_matrix(y_true,y_pred)
sns.heatmap(cm,annot=True,fmt='d',cmap='Blues')
plt.show()

In [ ]:
print("ROC AUC:",roc_auc_score(y_true,y_prob))

In [ ]:
from tensorflow.keras.preprocessing import image
import numpy as np
import matplotlib.pyplot as plt
import os

# User enters the image path
img_path = input("Enter the full image path: ").strip()

# Check if the file exists
if not os.path.exists(img_path):
    print("❌ Image not found. Please check the path.")
else:
    # Load image
    img = image.load_img(img_path, target_size=(224, 224))

    # Display image
    plt.imshow(img)
    plt.title("Selected X-ray")
    plt.axis("off")
    plt.show()

    # Preprocess image
    x = image.img_to_array(img)
    x = x / 255.0
    x = np.expand_dims(x, axis=0)

    # Predict
    prediction = model.predict(x, verbose=0)

    probability = prediction[0][0]

    if probability > 0.5:
        print("\n🩺 Prediction : Pneumonia")
        print(f"Confidence : {probability * 100:.2f}%")
    else:
        print("\n✅ Prediction : Normal")
        print(f"Confidence : {(1 - probability) * 100:.2f}%")